## Install Required Libraries

In [121]:
!pip install bert-score sacrebleu nltk sentencepiece -q

## Import Libraries

In [122]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

import nltk
from nltk.tokenize import word_tokenize

from collections import Counter

from sklearn.model_selection import train_test_split

import random
import time

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## Load the CSV Files

In [123]:
train_sa = pd.read_csv("train_sa_10000.csv")
train_en = pd.read_csv("train_en_10000.csv")

dev_sa = pd.read_csv("dev_sa_1000.csv")
dev_en = pd.read_csv("dev_en_1000.csv")

test_sa = pd.read_csv("test_sa_1000.csv")
test_en = pd.read_csv("test_en_1000.csv")

In [124]:
print(train_sa.head())
print()

print(train_en.head())

   Source_id                                        Sentence_sa
0          1                         "Ctrl, S नुत्वा रक्षन्तु।"
1          2                     गुरुः छात्रान् एकवारं पाठयति ।
2          3  चित्रचालनमिदं पुनः कर्तुं मया अस्याः राशेः चित...
3          4       वयं  Colors विकल्पं तस्योपरि नोदनेन चिनुमः ।
4          5  "अत्र कानिचन उदाहरणानि पश्याम:- एक: पर्वत:, चत...

   Source_id                                        Sentence_en
0          1                              Save it with Ctrl, S.
1          2         Teacher will teach the students only once.
2          3  To recreate this animation, I have to take two...
3          4    I will choose Colors options by clicking on it.
4          5  "See the example here - one mountain, four vil...


## Merge Sanskrit and English Files

In [125]:
train = train_sa.merge(train_en,on="Source_id")

dev = dev_sa.merge(dev_en,on="Source_id")

test = test_sa.merge(test_en,on="Source_id")

In [126]:
train.head()

,Source_id,Sentence_sa,Sentence_en
0,1,"""Ctrl, S नुत्वा रक्षन्तु।""","Save it with Ctrl, S."
1,2,गुरुः छात्रान् एकवारं पाठयति ।,Teacher will teach the students only once.
2,3,चित्रचालनमिदं पुनः कर्तुं मया अस्याः राशेः चित...,"To recreate this animation, I have to take two..."
3,4,वयं Colors विकल्पं तस्योपरि नोदनेन चिनुमः ।,I will choose Colors options by clicking on it.
4,5,"""अत्र कानिचन उदाहरणानि पश्याम:- एक: पर्वत:, चत...","""See the example here - one mountain, four vil..."


In [127]:
print("Training:",len(train))
print("Validation:",len(dev))
print("Testing:",len(test))

Training: 10000
Validation: 1000
Testing: 1000


In [128]:
import re

def clean_text(text):
    text = str(text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # Remove surrounding quotes
    text = text.strip('"')

    return text.strip()

train["Sentence_sa"] = train["Sentence_sa"].apply(clean_text)
train["Sentence_en"] = train["Sentence_en"].apply(clean_text)

dev["Sentence_sa"] = dev["Sentence_sa"].apply(clean_text)
dev["Sentence_en"] = dev["Sentence_en"].apply(clean_text)

test["Sentence_sa"] = test["Sentence_sa"].apply(clean_text)
test["Sentence_en"] = test["Sentence_en"].apply(clean_text)

In [129]:
train.head()

,Source_id,Sentence_sa,Sentence_en
0,1,"Ctrl, S नुत्वा रक्षन्तु।","Save it with Ctrl, S."
1,2,गुरुः छात्रान् एकवारं पाठयति ।,Teacher will teach the students only once.
2,3,चित्रचालनमिदं पुनः कर्तुं मया अस्याः राशेः चित...,"To recreate this animation, I have to take two..."
3,4,वयं Colors विकल्पं तस्योपरि नोदनेन चिनुमः ।,I will choose Colors options by clicking on it.
4,5,"अत्र कानिचन उदाहरणानि पश्याम:- एक: पर्वत:, चत्...","See the example here - one mountain, four vill..."


Build Tokenizers

In [130]:
def tokenize(text):
    return text.lower().split()

In [131]:
print(tokenize(train.iloc[0]["Sentence_sa"]))
print(tokenize(train.iloc[0]["Sentence_en"]))

['ctrl,', 's', 'नुत्वा', 'रक्षन्तु।']
['save', 'it', 'with', 'ctrl,', 's.']


Build the Vocabulary

In [132]:
from collections import Counter

sa_counter = Counter()
en_counter = Counter()

for sent in train["Sentence_sa"]:
    sa_counter.update(tokenize(sent))

for sent in train["Sentence_en"]:
    en_counter.update(tokenize(sent))

In [133]:
SPECIAL_TOKENS = [
    "<PAD>",
    "<SOS>",
    "<EOS>",
    "<UNK>"
]

Create Sanskrit Vocabulary

In [134]:
sa_vocab = SPECIAL_TOKENS + sorted(sa_counter.keys())

sa_word2idx = {word: idx for idx, word in enumerate(sa_vocab)}
sa_idx2word = {idx: word for word, idx in sa_word2idx.items()}

Create English Vocabulary

In [135]:
en_vocab = SPECIAL_TOKENS + sorted(en_counter.keys())

en_word2idx = {word: idx for idx, word in enumerate(en_vocab)}
en_idx2word = {idx: word for word, idx in en_word2idx.items()}

Check Vocabulary Sizes

In [136]:
print("Sanskrit Vocabulary:", len(sa_vocab))
print("English Vocabulary:", len(en_vocab))

Sanskrit Vocabulary: 32275
English Vocabulary: 16661


In [137]:
print(list(sa_word2idx.items())[:10])
print()
print(list(en_word2idx.items())[:10])

[('<PAD>', 0), ('<SOS>', 1), ('<EOS>', 2), ('<UNK>', 3), ('!', 4), ('!p', 5), ('!नानाप्रकारेषु', 6), ('"', 7), ('""', 8), ('""अद्य', 9)]

[('<PAD>', 0), ('<SOS>', 1), ('<EOS>', 2), ('<UNK>', 3), ('!', 4), ('!clears', 5), ('!p', 6), ('"', 7), ('""', 8), ('""accomplished,""', 9)]


Convert Sentences to Token IDs

In [138]:
# Encode Sentences
MAX_LEN = 50

def encode(sentence, vocab):
    tokens = tokenize(sentence)

    ids = [vocab["<SOS>"]]

    for token in tokens:
        ids.append(vocab.get(token, vocab["<UNK>"]))

    ids.append(vocab["<EOS>"])

    return ids[:MAX_LEN]

In [139]:
# Padding Function
def pad_sequence(seq, max_len=MAX_LEN):
    if len(seq) < max_len:
        seq += [0] * (max_len - len(seq))
    else:
        seq = seq[:max_len]

    return seq

In [140]:
# Encode Entire Dataset
train_sa_ids = [
    pad_sequence(encode(x, sa_word2idx))
    for x in train["Sentence_sa"]
]

train_en_ids = [
    pad_sequence(encode(x, en_word2idx))
    for x in train["Sentence_en"]
]

dev_sa_ids = [
    pad_sequence(encode(x, sa_word2idx))
    for x in dev["Sentence_sa"]
]

dev_en_ids = [
    pad_sequence(encode(x, en_word2idx))
    for x in dev["Sentence_en"]
]

In [141]:
# Create a Dataset Class
from torch.utils.data import Dataset

class TranslationDataset(Dataset):

    def __init__(self, src, tgt):

        self.src = src
        self.tgt = tgt

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):

        return (
            torch.tensor(self.src[idx]),
            torch.tensor(self.tgt[idx])
        )

In [142]:
# Create Dataset Objects
train_dataset = TranslationDataset(
    train_sa_ids,
    train_en_ids
)

dev_dataset = TranslationDataset(
    dev_sa_ids,
    dev_en_ids
)

In [143]:
# Create DataLoaders
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [144]:
src, tgt = next(iter(train_loader))

print(src.shape)
print(tgt.shape)

torch.Size([64, 50])
torch.Size([64, 50])


Build the Encoder

In [145]:
# Select Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)


cuda


In [146]:

INPUT_DIM = len(sa_vocab)
OUTPUT_DIM = len(en_vocab)

ENC_EMB_DIM = 256
DEC_EMB_DIM = 256

HIDDEN_DIM = 512

NUM_LAYERS = 2

DROPOUT = 0.3

In [147]:
import torch.nn as nn

class Encoder(nn.Module):

    def __init__(self,
                 input_dim,
                 emb_dim,
                 hidden_dim,
                 num_layers,
                 dropout):

        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim)

        self.rnn = nn.LSTM(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True,
            bidirectional=True
        )

        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_cell = nn.Linear(hidden_dim * 2, hidden_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, src):

        embedded = self.dropout(self.embedding(src))

        outputs, (hidden, cell) = self.rnn(embedded)

        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        cell = torch.cat((cell[-2], cell[-1]), dim=1)

        hidden = torch.tanh(self.fc_hidden(hidden)).unsqueeze(0)
        cell = torch.tanh(self.fc_cell(cell)).unsqueeze(0)

        return outputs, hidden, cell

In [148]:
encoder = Encoder(
    INPUT_DIM,
    ENC_EMB_DIM,
    HIDDEN_DIM,
    NUM_LAYERS,
    DROPOUT
).to(device)

In [149]:
src, tgt = next(iter(train_loader))

src = src.to(device)

outputs, hidden, cell = encoder(src)

print(outputs.shape)
print(hidden.shape)
print(cell.shape)

torch.Size([64, 50, 1024])
torch.Size([1, 64, 512])
torch.Size([1, 64, 512])


In [150]:
class Attention(nn.Module):

    def __init__(self, hidden_dim):

        super().__init__()

        self.attn = nn.Linear(hidden_dim * 3, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):

        # hidden: [1, batch, hidden]
        # encoder_outputs: [batch, src_len, hidden*2]

        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]

        hidden = hidden.permute(1, 0, 2)
        hidden = hidden.repeat(1, src_len, 1)

        energy = torch.tanh(
            self.attn(
                torch.cat((hidden, encoder_outputs), dim=2)
            )
        )

        attention = self.v(energy).squeeze(2)

        return torch.softmax(attention, dim=1)

In [151]:
attention = Attention(HIDDEN_DIM).to(device)

In [152]:
weights = attention(hidden, outputs)

print(weights.shape)

torch.Size([64, 50])


In [153]:
print(weights[0].sum())

tensor(1., device='cuda:0', grad_fn=<SumBackward0>)


Build the Decoder

In [154]:
class Decoder(nn.Module):

    def __init__(
        self,
        output_dim,
        emb_dim,
        hidden_dim,
        num_layers,
        dropout,
        attention
    ):

        super().__init__()

        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.attention = attention

        self.embedding = nn.Embedding(output_dim, emb_dim)

        self.rnn = nn.LSTM(
            emb_dim + hidden_dim * 2,
            hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.fc_out = nn.Linear(
            emb_dim + hidden_dim * 3,
            output_dim
        )

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        input,
        hidden,
        cell,
        encoder_outputs
    ):

        input = input.unsqueeze(1)

        embedded = self.dropout(
            self.embedding(input)
        )

        attention_weights = self.attention(
            hidden,
            encoder_outputs
        )

        attention_weights = attention_weights.unsqueeze(1)

        context = torch.bmm(
            attention_weights,
            encoder_outputs
        )

        rnn_input = torch.cat(
            (embedded, context),
            dim=2
        )

        output, (hidden, cell) = self.rnn(
            rnn_input,
            (hidden, cell)
        )

        prediction = self.fc_out(
            torch.cat(
                (
                    output.squeeze(1),
                    context.squeeze(1),
                    embedded.squeeze(1)
                ),
                dim=1
            )
        )

        return prediction, hidden, cell, attention_weights

In [155]:
decoder = Decoder(
    OUTPUT_DIM,
    DEC_EMB_DIM,
    HIDDEN_DIM,
    NUM_LAYERS,
    DROPOUT,
    attention
).to(device)

In [156]:
src, tgt = next(iter(train_loader))

src = src.to(device)
tgt = tgt.to(device)

encoder_outputs, hidden, cell = encoder(src)

input_word = tgt[:, 0]

prediction, hidden, cell, attention = decoder(
    input_word,
    hidden,
    cell,
    encoder_outputs
)

print(prediction.shape)
print(hidden.shape)
print(cell.shape)
print(attention.shape)

torch.Size([64, 16661])
torch.Size([1, 64, 512])
torch.Size([1, 64, 512])
torch.Size([64, 1, 50])


Build the Seq2Seq Model

In [157]:
class Seq2Seq(nn.Module):

    def __init__(self, encoder, decoder, device):

        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(
        self,
        src,
        trg,
        teacher_forcing_ratio=0.5
    ):

        batch_size = trg.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(
            batch_size,
            trg_len,
            trg_vocab_size
        ).to(self.device)

        encoder_outputs, hidden, cell = self.encoder(src)

        input = trg[:, 0]

        for t in range(1, trg_len):

            output, hidden, cell, _ = self.decoder(
                input,
                hidden,
                cell,
                encoder_outputs
            )

            outputs[:, t] = output

            top1 = output.argmax(1)

            teacher_force = random.random() < teacher_forcing_ratio

            input = trg[:, t] if teacher_force else top1

        return outputs

In [158]:
model = Seq2Seq(
    encoder,
    decoder,
    device
).to(device)

In [159]:
src, tgt = next(iter(train_loader))

src = src.to(device)
tgt = tgt.to(device)

outputs = model(src, tgt)

print(outputs.shape)

torch.Size([64, 50, 16661])


Loss Function & Optimizer

In [160]:
criterion = nn.CrossEntropyLoss(
    ignore_index=en_word2idx["<PAD>"]
)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [161]:
def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

print("Trainable Parameters:", count_parameters(model))

Trainable Parameters: 57365525


Training the Model

In [162]:
import torch

def train(model, loader, optimizer, criterion, clip):

    model.train()

    epoch_loss = 0

    for src, trg in loader:

        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()

        output = model(src, trg)

        # output = [batch, trg_len, vocab_size]
        output = output[:, 1:, :].reshape(-1, OUTPUT_DIM)

        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(loader)

In [163]:
def evaluate(model, loader, criterion):

    model.eval()

    epoch_loss = 0

    with torch.no_grad():

        for src, trg in loader:

            src = src.to(device)
            trg = trg.to(device)

            output = model(src, trg, teacher_forcing_ratio=0)

            output = output[:, 1:, :].reshape(-1, OUTPUT_DIM)

            trg = trg[:, 1:].reshape(-1)

            loss = criterion(output, trg)

            epoch_loss += loss.item()

    return epoch_loss / len(loader)

In [164]:
N_EPOCHS = 20
CLIP = 1

best_valid_loss = float("inf")

for epoch in range(N_EPOCHS):

    train_loss = train(
        model,
        train_loader,
        optimizer,
        criterion,
        CLIP
    )

    valid_loss = evaluate(
        model,
        dev_loader,
        criterion
    )

    if valid_loss < best_valid_loss:

        best_valid_loss = valid_loss

        torch.save(
            model.state_dict(),
            "best_model.pt"
        )

    print(f"Epoch {epoch+1}")

    print(f"Train Loss : {train_loss:.4f}")

    print(f"Valid Loss : {valid_loss:.4f}")

    print("-"*50)

Epoch 1
Train Loss : 6.9194
Valid Loss : 6.9432
--------------------------------------------------
Epoch 2
Train Loss : 6.3077
Valid Loss : 6.8941
--------------------------------------------------
Epoch 3
Train Loss : 5.9470
Valid Loss : 6.8825
--------------------------------------------------
Epoch 4
Train Loss : 5.6321
Valid Loss : 6.9097
--------------------------------------------------
Epoch 5
Train Loss : 5.3530
Valid Loss : 6.9112
--------------------------------------------------
Epoch 6
Train Loss : 5.1163
Valid Loss : 6.9700
--------------------------------------------------
Epoch 7
Train Loss : 4.9165
Valid Loss : 6.9585
--------------------------------------------------
Epoch 8
Train Loss : 4.6923
Valid Loss : 7.0487
--------------------------------------------------
Epoch 9
Train Loss : 4.4749
Valid Loss : 7.0246
--------------------------------------------------
Epoch 10
Train Loss : 4.2341
Valid Loss : 7.1190
--------------------------------------------------
Epoch 11


In [165]:
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(32275, 256)
    (rnn): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
    (fc_hidden): Linear(in_features=1024, out_features=512, bias=True)
    (fc_cell): Linear(in_features=1024, out_features=512, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (decoder): Decoder(
    (attention): Attention(
      (attn): Linear(in_features=1536, out_features=512, bias=True)
      (v): Linear(in_features=512, out_features=1, bias=False)
    )
    (embedding): Embedding(16661, 256)
    (rnn): LSTM(1280, 512, batch_first=True)
    (fc_out): Linear(in_features=1792, out_features=16661, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
)

In [166]:
import torch
import torch.nn.functional as F

def beam_search(model,
                sentence,
                beam_width=5,
                max_len=50):

    model.eval()

    # Encode source sentence
    src = torch.tensor(
        [pad_sequence(encode(sentence, sa_word2idx))]
    ).to(device)

    with torch.no_grad():

        encoder_outputs, hidden, cell = model.encoder(src)

    beams = [(
        [en_word2idx["<SOS>"]],
        0.0,
        hidden,
        cell
    )]

    completed = []

    for _ in range(max_len):

        candidates = []

        for seq, score, h, c in beams:

            if seq[-1] == en_word2idx["<EOS>"]:

                completed.append((seq, score))
                continue

            inp = torch.tensor(
                [seq[-1]]
            ).to(device)

            with torch.no_grad():

                output, h_new, c_new, _ = model.decoder(
                    inp,
                    h,
                    c,
                    encoder_outputs
                )

            log_probs = F.log_softmax(output, dim=1)

            values, indices = torch.topk(
                log_probs,
                beam_width
            )

            for k in range(beam_width):

                next_word = indices[0][k].item()

                next_score = score + values[0][k].item()

                candidates.append((
                    seq + [next_word],
                    next_score,
                    h_new,
                    c_new
                ))

        candidates = sorted(
            candidates,
            key=lambda x: x[1],
            reverse=True
        )

        beams = candidates[:beam_width]

    if completed:

        best = max(completed, key=lambda x: x[1])[0]

    else:

        best = beams[0][0]

    words = []

    for idx in best:

        word = en_idx2word[idx]

        if word == "<EOS>":
            break

        if word not in ["<SOS>", "<PAD>"]:

            words.append(word)

    return " ".join(words)

In [167]:
sample = test.iloc[0]["Sentence_sa"]

print(sample)

एक्लिप्स् इति प्रोग्रामर् कृते दोषान्वेषणे अपि साहाय्यं करोति।


In [168]:
prediction = beam_search(
    model,
    sample,
    beam_width=5
)

print(prediction)

in this tutorial, i am using: ubuntu linux os 16.04


In [169]:
predictions = []

for sentence in test["Sentence_sa"]:

    pred = beam_search(
        model,
        sentence,
        beam_width=5
    )

    predictions.append(pred)

In [170]:
submission = test[["Source_id"]].copy()

submission["Sentence_en"] = predictions

submission.to_csv(
    "submission.csv",
    index=False,
    encoding="utf-8"
)

print(submission.head())

   Source_id                                        Sentence_en
0          1  in this tutorial, i am using: ubuntu linux os ...
1          2  and when i say unto you, to the and of the and...
2          3  and when he was the the of the and of the and ...
3          4  this is the the of the of the and of the and o...
4          5  and when he said unto them, and he was the and...


In [171]:
def translate_sentence(model, sentence, max_len=50):

    model.eval()

    src = torch.tensor(
        [pad_sequence(encode(sentence, sa_word2idx))]
    ).to(device)

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src)

    outputs = [en_word2idx["<SOS>"]]

    for _ in range(max_len):

        inp = torch.tensor([outputs[-1]]).to(device)

        with torch.no_grad():
            prediction, hidden, cell, _ = model.decoder(
                inp,
                hidden,
                cell,
                encoder_outputs
            )

        best = prediction.argmax(1).item()

        if best == en_word2idx["<EOS>"]:
            break

        outputs.append(best)

    words = []

    for idx in outputs[1:]:
        words.append(en_idx2word[idx])

    return " ".join(words)

In [172]:
print(translate_sentence(model, test.iloc[0]["Sentence_sa"]))

the the the the the the the the


In [173]:
from nltk.translate.bleu_score import corpus_bleu

references = []
hypotheses = []

for i in range(len(dev)):

    src = dev.iloc[i]["Sentence_sa"]

    ref = dev.iloc[i]["Sentence_en"].lower().split()

    pred = translate_sentence(model, src).lower().split()

    references.append([ref])
    hypotheses.append(pred)

bleu = corpus_bleu(references, hypotheses)

print("BLEU Score:", bleu)

BLEU Score: 0.004576409224787604


In [174]:
from bert_score import score

predictions = []
references = []

for i in range(len(dev)):

    src = dev.iloc[i]["Sentence_sa"]

    pred = translate_sentence(model, src)

    predictions.append(pred)

    references.append(dev.iloc[i]["Sentence_en"])

P, R, F1 = score(
    predictions,
    references,
    lang="en",
    rescale_with_baseline=True
)

print("Average BERTScore:", F1.mean().item())

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Average BERTScore: -0.19574390351772308


In [175]:
import time

start = time.time()

predictions = []

for sentence in test["Sentence_sa"]:

    pred = beam_search(
        model,
        sentence,
        beam_width=5
    )

    predictions.append(pred)

end = time.time()

print("Inference Time:", end - start, "seconds")

Inference Time: 233.9224100112915 seconds


In [176]:
submission = test[["Source_id"]].copy()

submission["Sentence_en"] = predictions

submission.to_csv(
    "submission.csv",
    index=False,
    encoding="utf-8"
)

submission.head()

,Source_id,Sentence_en
0,1,"in this tutorial, i am using: ubuntu linux os ..."
1,2,"and when i say unto you, to the and of the and..."
2,3,and when he was the the of the and of the and ...
3,4,this is the the of the of the and of the and o...
4,5,"and when he said unto them, and he was the and..."
